In [1]:
from MAGE_utils import *

/opt/homebrew/Caskroom/miniconda/base/envs/testenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
testbed1, testbed2, testbed3, testbed4, testbed5, testbed6, testbed7, testbed8 = process_dataset(load_testbed1_data()), process_dataset(load_testbed2_data()), process_dataset(load_testbed3_data()), process_dataset(load_testbed4_data()), process_dataset(load_testbed5_data()), process_dataset(load_testbed6_data()), process_dataset(load_testbed7_data()), process_dataset(load_testbed8_data())

In [3]:
for k, v in testbed7.items():
    testbed4[k] = v
for k, v in testbed8.items():
    testbed4[k] = v

In [43]:
# ============================================
# Unified Prototype-Augmented Hierarchical Contrastive Detection
# Adapted for multiple MAGE-style testbeds
#
# Supports:
# 1) A single DatasetDict:
#       train / validation / test
#       or train / validation / test_id / test_ood
#       or train / validation / test / test_ood_gpt / test_ood_gpt_para
#
# 2) A dict[str, DatasetDict], e.g.:
#       {"cmv": DatasetDict(...), "eli5": DatasetDict(...), ...}
#
# Inference:
#   A) prototype score
#   B) classifier-head score
#
# Thresholds are selected on validation set independently for A and B.
# ============================================

import random
import numpy as np
from collections import Counter
from typing import Dict, List, Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


# =========================================================
# 0) Global Config
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_NAME = "roberta-base"
MAX_LEN = 256
BATCH_SIZE = 32 if device == "cuda" else 8

EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

TAU = 0.07
LAMBDA_FAMILY = 1.0
BETA_CE = 1.0

HUMAN_LABEL = 1
AI_LABEL = 0
HUMAN_MODEL_LABEL = 0

TEXT_FIELD = "text"
LABEL_FIELD = "label"
MODEL_LABEL_FIELD = "model_label"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# =========================================================
# 1) Utilities for dataset structure
# =========================================================
def is_datasetdict_like(obj):
    return hasattr(obj, "keys") and "train" in obj and "validation" in obj

def is_mapping_of_datasetdict(obj):
    if not hasattr(obj, "items"):
        return False
    if len(obj) == 0:
        return False
    first_val = next(iter(obj.values()))
    return is_datasetdict_like(first_val)

def available_eval_splits(ds_dict) -> List[str]:
    """
    Dynamically detect available evaluation splits.
    Priority order is only for prettier printing.
    """
    candidate_order = [
        "test",
        "test_id",
        "test_ood",
        "test_ood_gpt",
        "test_ood_gpt_para",
    ]
    return [sp for sp in candidate_order if sp in ds_dict]

def print_split_counters(ds_dict, name="dataset"):
    print(f"\n========== {name} ==========")
    for split in ds_dict.keys():
        if LABEL_FIELD in ds_dict[split].column_names:
            print(f"{split} label counter:", Counter(ds_dict[split][LABEL_FIELD]))
        if MODEL_LABEL_FIELD in ds_dict[split].column_names:
            print(f"{split} model_label counter:", Counter(ds_dict[split][MODEL_LABEL_FIELD]))


# =========================================================
# 2) Dynamic machine model labels from TRAIN SET
# =========================================================
def infer_machine_model_labels(train_ds):
    """
    Infer actually existing machine model_label values from train set.

    Rule:
      - human samples: label == 1, model_label == 0
      - machine samples: label == 0, model_label != 0
    """
    labels = train_ds[LABEL_FIELD]
    model_labels = train_ds[MODEL_LABEL_FIELD]

    machine_model_labels = sorted({
        int(m)
        for y, m in zip(labels, model_labels)
        if int(y) == AI_LABEL and int(m) != HUMAN_MODEL_LABEL
    })
    return machine_model_labels

def validate_label_consistency(train_ds, strict=False):
    """
    Optional sanity check:
      - human sample should have model_label == 0
      - machine sample should have model_label != 0
    """
    labels = train_ds[LABEL_FIELD]
    model_labels = train_ds[MODEL_LABEL_FIELD]

    bad_human = []
    bad_machine = []

    for i, (y, m) in enumerate(zip(labels, model_labels)):
        y = int(y)
        m = int(m)
        if y == HUMAN_LABEL and m != HUMAN_MODEL_LABEL:
            bad_human.append(i)
        if y == AI_LABEL and m == HUMAN_MODEL_LABEL:
            bad_machine.append(i)

    if len(bad_human) > 0 or len(bad_machine) > 0:
        msg = (
            f"[Warning] Inconsistent labels found. "
            f"human-with-nonzero-model_label={len(bad_human)}, "
            f"machine-with-zero-model_label={len(bad_machine)}"
        )
        if strict:
            raise ValueError(msg)
        print(msg)


# =========================================================
# 3) DataLoader
# =========================================================
def collate_fn(batch):
    texts = [b[TEXT_FIELD] for b in batch]
    labels = torch.tensor([b[LABEL_FIELD] for b in batch], dtype=torch.long)
    model_labels = torch.tensor([b[MODEL_LABEL_FIELD] for b in batch], dtype=torch.long)

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    return enc, labels, model_labels

def make_loader(ds, shuffle=False):
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_fn,
        drop_last=False
    )


# =========================================================
# 4) Model
# =========================================================
class PrototypeContrastiveDetector(nn.Module):
    def __init__(self, model_name, machine_model_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden = self.encoder.config.hidden_size

        self.cls_head = nn.Linear(self.hidden, 2)

        self.machine_model_labels = list(machine_model_labels)
        self.num_machine_families = len(self.machine_model_labels)
        self.model_label_to_index = {
            ml: idx for idx, ml in enumerate(self.machine_model_labels)
        }
        self.index_to_model_label = {
            idx: ml for ml, idx in self.model_label_to_index.items()
        }

        self.human_prototype = nn.Parameter(torch.randn(self.hidden))

        if self.num_machine_families > 0:
            self.machine_prototypes = nn.Parameter(
                torch.randn(self.num_machine_families, self.hidden)
            )
        else:
            self.machine_prototypes = nn.Parameter(torch.empty(0, self.hidden))

    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp_min(1e-9)
        return summed / denom

    def encode(self, enc, normalize=True):
        out = self.encoder(**enc)
        z = self.mean_pool(out.last_hidden_state, enc["attention_mask"])
        if normalize:
            z = F.normalize(z, dim=-1)
        return z

    def forward(self, enc):
        z = self.encode(enc, normalize=True)
        logits = self.cls_head(z)
        return z, logits


# =========================================================
# 5) Prototype initialization
# =========================================================
@torch.no_grad()
def initialize_prototypes_from_trainset(model, loader):
    model.eval()

    d = model.hidden
    K = model.num_machine_families

    human_sum = torch.zeros(d, device=device)
    human_cnt = 0

    machine_sum = torch.zeros(K, d, device=device)
    machine_cnt = torch.zeros(K, device=device)

    for enc, labels, model_labels in tqdm(loader, desc="Initializing prototypes"):
        enc = {k: v.to(device) for k, v in enc.items()}
        labels = labels.to(device)
        model_labels = model_labels.to(device)

        z = model.encode(enc, normalize=False)

        hmask = (labels == HUMAN_LABEL)
        if hmask.any():
            human_sum += z[hmask].sum(dim=0)
            human_cnt += int(hmask.sum().item())

        for ml, idx in model.model_label_to_index.items():
            kmask = (model_labels == ml)
            if kmask.any():
                machine_sum[idx] += z[kmask].sum(dim=0)
                machine_cnt[idx] += int(kmask.sum().item())

    if human_cnt == 0:
        raise ValueError("No human samples found in train set.")
    model.human_prototype.data.copy_(human_sum / human_cnt)

    if K > 0:
        machine_prototypes = []
        for idx, ml in enumerate(model.machine_model_labels):
            if machine_cnt[idx] == 0:
                raise ValueError(f"No machine samples found for machine model_label {ml}.")
            machine_prototypes.append(machine_sum[idx] / machine_cnt[idx])
        machine_prototypes = torch.stack(machine_prototypes, dim=0)
        model.machine_prototypes.data.copy_(machine_prototypes)

    print("Initialized human prototype from", human_cnt, "human samples")
    print("Machine model labels:", model.machine_model_labels)
    if K > 0:
        print("Counts per machine prototype:", machine_cnt.long().tolist())
    else:
        print("No machine prototype initialized because no machine model_label exists in train set.")


# =========================================================
# 6) Contrastive loss utilities
# =========================================================
def positive_average_contrastive_loss(anchor, pos_list, neg_list, tau=0.07):
    """
    - (1/|P|) sum_{p in P} log( exp(sim(a,p)/tau) / sum_{u in P∪N} exp(sim(a,u)/tau) )
    """
    anchor = F.normalize(anchor.unsqueeze(0), dim=-1).squeeze(0)

    pos_sims = []
    for p in pos_list:
        p = F.normalize(p.unsqueeze(0), dim=-1).squeeze(0)
        pos_sims.append(torch.dot(anchor, p) / tau)

    neg_sims = []
    for n in neg_list:
        n = F.normalize(n.unsqueeze(0), dim=-1).squeeze(0)
        neg_sims.append(torch.dot(anchor, n) / tau)

    pos_sims = torch.stack(pos_sims)
    all_sims = torch.cat([pos_sims, torch.stack(neg_sims)], dim=0)

    log_denom = torch.logsumexp(all_sims, dim=0)
    loss = -(pos_sims.mean() - log_denom)
    return loss

def build_batch_groups(labels, model_labels, machine_model_labels):
    human_idx = (labels == HUMAN_LABEL).nonzero(as_tuple=False).squeeze(-1)
    machine_idx = (labels == AI_LABEL).nonzero(as_tuple=False).squeeze(-1)

    family_to_idx = {}
    for ml in machine_model_labels:
        family_to_idx[ml] = (model_labels == ml).nonzero(as_tuple=False).squeeze(-1)

    return human_idx, machine_idx, family_to_idx


# =========================================================
# 7) Hierarchical prototype-augmented contrastive loss
# =========================================================
def compute_contrastive_loss(model, z, labels, model_labels, tau=0.07, lambda_family=1.0):
    human_prototype = model.human_prototype
    machine_prototypes = model.machine_prototypes
    machine_model_labels = model.machine_model_labels
    K = model.num_machine_families

    human_idx, machine_idx, family_to_idx = build_batch_groups(
        labels, model_labels, machine_model_labels
    )

    has_family_contrast = K >= 2
    losses = []

    # ---------- a) Human samples ----------
    for i in human_idx.tolist():
        pos_list = []
        neg_list = []

        for j in human_idx.tolist():
            if j != i:
                pos_list.append(z[j])
        pos_list.append(human_prototype)

        for j in machine_idx.tolist():
            neg_list.append(z[j])
        for k in range(K):
            neg_list.append(machine_prototypes[k])

        if len(pos_list) > 0 and len(neg_list) > 0:
            losses.append(
                positive_average_contrastive_loss(z[i], pos_list, neg_list, tau=tau)
            )

    # ---------- b) Human prototype ----------
    pos_list = []
    neg_list = []

    for j in human_idx.tolist():
        pos_list.append(z[j])
    for j in machine_idx.tolist():
        neg_list.append(z[j])
    for k in range(K):
        neg_list.append(machine_prototypes[k])

    if len(pos_list) > 0 and len(neg_list) > 0:
        losses.append(
            positive_average_contrastive_loss(human_prototype, pos_list, neg_list, tau=tau)
        )

    # ---------- c) Machine samples ----------
    for ml in machine_model_labels:
        family_idx = family_to_idx[ml]
        if family_idx.numel() == 0:
            continue

        proto_idx = model.model_label_to_index[ml]

        for i in family_idx.tolist():
            # global contrast
            pos_list_g = []
            neg_list_g = []

            for j in machine_idx.tolist():
                if j != i:
                    pos_list_g.append(z[j])
            for k in range(K):
                pos_list_g.append(machine_prototypes[k])

            for j in human_idx.tolist():
                neg_list_g.append(z[j])
            neg_list_g.append(human_prototype)

            if len(pos_list_g) > 0 and len(neg_list_g) > 0:
                loss_g = positive_average_contrastive_loss(
                    z[i], pos_list_g, neg_list_g, tau=tau
                )
            else:
                loss_g = z.sum() * 0.0

            if has_family_contrast:
                pos_list_f = []
                neg_list_f = []

                for j in family_idx.tolist():
                    if j != i:
                        pos_list_f.append(z[j])
                pos_list_f.append(machine_prototypes[proto_idx])

                for other_ml in machine_model_labels:
                    if other_ml == ml:
                        continue
                    other_idx = model.model_label_to_index[other_ml]
                    for j in family_to_idx[other_ml].tolist():
                        neg_list_f.append(z[j])
                    neg_list_f.append(machine_prototypes[other_idx])

                if len(pos_list_f) > 0 and len(neg_list_f) > 0:
                    loss_f = positive_average_contrastive_loss(
                        z[i], pos_list_f, neg_list_f, tau=tau
                    )
                else:
                    loss_f = z.sum() * 0.0

                losses.append(loss_g + lambda_family * loss_f)
            else:
                losses.append(loss_g)

    # ---------- d) Machine prototypes ----------
    for ml in machine_model_labels:
        proto_idx = model.model_label_to_index[ml]

        pos_list_g = []
        neg_list_g = []

        for j in machine_idx.tolist():
            pos_list_g.append(z[j])
        for other_ml in machine_model_labels:
            if other_ml != ml:
                other_idx = model.model_label_to_index[other_ml]
                pos_list_g.append(machine_prototypes[other_idx])

        for j in human_idx.tolist():
            neg_list_g.append(z[j])
        neg_list_g.append(human_prototype)

        if len(pos_list_g) > 0 and len(neg_list_g) > 0:
            loss_g = positive_average_contrastive_loss(
                machine_prototypes[proto_idx], pos_list_g, neg_list_g, tau=tau
            )
        else:
            loss_g = z.sum() * 0.0

        if has_family_contrast:
            pos_list_f = []
            neg_list_f = []

            for j in family_to_idx[ml].tolist():
                pos_list_f.append(z[j])

            for other_ml in machine_model_labels:
                if other_ml == ml:
                    continue
                other_idx = model.model_label_to_index[other_ml]
                for j in family_to_idx[other_ml].tolist():
                    neg_list_f.append(z[j])
                neg_list_f.append(machine_prototypes[other_idx])

            if len(pos_list_f) > 0 and len(neg_list_f) > 0:
                loss_f = positive_average_contrastive_loss(
                    machine_prototypes[proto_idx], pos_list_f, neg_list_f, tau=tau
                )
            else:
                loss_f = z.sum() * 0.0

            losses.append(loss_g + lambda_family * loss_f)
        else:
            losses.append(loss_g)

    if len(losses) == 0:
        return z.sum() * 0.0

    return torch.stack(losses).mean()


# =========================================================
# 8) Training / Inference / Evaluation
# =========================================================
def train_one_epoch(model, optimizer, train_loader, epoch_idx=0):
    model.train()
    total_loss = 0.0
    total_contrast = 0.0
    total_ce = 0.0
    steps = 0

    for enc, labels, model_labels in tqdm(train_loader, desc=f"Train epoch {epoch_idx}"):
        enc = {k: v.to(device) for k, v in enc.items()}
        labels = labels.to(device)
        model_labels = model_labels.to(device)

        z, logits = model(enc)

        contrast_loss = compute_contrastive_loss(
            model,
            z,
            labels,
            model_labels,
            tau=TAU,
            lambda_family=LAMBDA_FAMILY
        )

        ce_loss = F.cross_entropy(logits, labels)
        loss = contrast_loss + BETA_CE * ce_loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()
        total_contrast += contrast_loss.item()
        total_ce += ce_loss.item()
        steps += 1

    return {
        "loss": total_loss / max(steps, 1),
        "contrast": total_contrast / max(steps, 1),
        "ce": total_ce / max(steps, 1),
    }


@torch.no_grad()
def collect_prototype_scores(model, loader):
    model.eval()

    h_prototype = F.normalize(model.human_prototype.unsqueeze(0), dim=-1).squeeze(0)
    m_prototypes = F.normalize(model.machine_prototypes, dim=-1)

    all_scores = []
    all_y = []

    for enc, labels, model_labels in tqdm(loader, desc="Collect prototype scores"):
        enc = {k: v.to(device) for k, v in enc.items()}
        labels = labels.to(device)

        z = model.encode(enc, normalize=True)

        s_h = torch.matmul(z, h_prototype)

        if m_prototypes.size(0) > 0:
            s_m_all = torch.matmul(z, m_prototypes.t())
            s_m = torch.max(s_m_all, dim=1).values
        else:
            s_m = torch.zeros_like(s_h)

        score = s_h - s_m

        all_scores.append(score.detach().cpu().numpy())
        all_y.append(labels.detach().cpu().numpy())

    return np.concatenate(all_scores), np.concatenate(all_y)


@torch.no_grad()
def collect_cls_scores(model, loader):
    model.eval()

    all_scores = []
    all_y = []

    for enc, labels, model_labels in tqdm(loader, desc="Collect classifier-head scores"):
        enc = {k: v.to(device) for k, v in enc.items()}
        labels = labels.to(device)

        _, logits = model(enc)
        probs = F.softmax(logits, dim=-1)
        score_cls = probs[:, HUMAN_LABEL]

        all_scores.append(score_cls.detach().cpu().numpy())
        all_y.append(labels.detach().cpu().numpy())

    return np.concatenate(all_scores), np.concatenate(all_y)


def youden_best_threshold(scores, y):
    candidates = np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 600)))
    best_J, best_delta = -1e9, None

    is_h = (y == HUMAN_LABEL)
    is_a = (y == AI_LABEL)

    for delta in candidates:
        pred_h = (scores >= delta)
        TPR = pred_h[is_h].mean() if is_h.any() else 0.0
        FPR = pred_h[is_a].mean() if is_a.any() else 0.0
        J = TPR - FPR
        if J > best_J:
            best_J = J
            best_delta = float(delta)

    return best_delta, float(best_J)


def eval_metrics(scores, y, delta):
    pred = (scores >= delta).astype(int)

    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred, pos_label=HUMAN_LABEL)

    is_h = (y == HUMAN_LABEL)
    is_a = (y == AI_LABEL)

    human_recall = float(pred[is_h].mean()) if is_h.any() else 0.0
    ai_recall = float((1 - pred[is_a]).mean()) if is_a.any() else 0.0

    try:
        auroc = roc_auc_score(y, scores)
    except Exception:
        auroc = float("nan")

    return {
        "acc": float(acc),
        "human_recall": human_recall,
        "ai_recall": ai_recall,
        "f1": float(f1),
        "auroc": float(auroc),
    }


# =========================================================
# 9) Run one DatasetDict experiment
# =========================================================
def run_single_datasetdict_experiment(
    ds_dict,
    exp_name="experiment",
    model_name=MODEL_NAME,
    epochs=EPOCHS,
):
    """
    ds_dict can be any of:
      - train / validation / test
      - train / validation / test_id / test_ood
      - train / validation / test / test_ood_gpt / test_ood_gpt_para
    """
    print_split_counters(ds_dict, exp_name)

    validate_label_consistency(ds_dict["train"], strict=False)
    machine_model_labels = infer_machine_model_labels(ds_dict["train"])

    print("Inferred machine model labels from TRAIN:", machine_model_labels)
    print("Number of machine families:", len(machine_model_labels))

    train_loader = make_loader(ds_dict["train"], shuffle=True)
    train_noshuf_loader = make_loader(ds_dict["train"], shuffle=False)
    val_loader = make_loader(ds_dict["validation"], shuffle=False)

    eval_split_names = available_eval_splits(ds_dict)
    eval_loaders = {sp: make_loader(ds_dict[sp], shuffle=False) for sp in eval_split_names}

    model = PrototypeContrastiveDetector(model_name, machine_model_labels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    initialize_prototypes_from_trainset(model, train_noshuf_loader)

    for ep in range(epochs):
        stats = train_one_epoch(model, optimizer, train_loader, epoch_idx=ep)
        print(f"[{exp_name}] epoch {ep}: {stats}")

    results = {
        "meta": {
            "exp_name": exp_name,
            "machine_model_labels": machine_model_labels,
            "num_machine_families": len(machine_model_labels),
            "eval_splits": eval_split_names,
        },
        "prototype": {},
        "classifier": {},
    }

    # ---------- prototype branch ----------
    val_scores_proto, val_y_proto = collect_prototype_scores(model, val_loader)
    delta_proto, J_proto = youden_best_threshold(val_scores_proto, val_y_proto)

    results["prototype"]["validation"] = {
        "threshold": delta_proto,
        "J": J_proto,
        **eval_metrics(val_scores_proto, val_y_proto, delta_proto),
    }

    print(f"[{exp_name}][VAL][PROTO] best delta = {delta_proto:.6f}, J = {J_proto:.4f}")
    print(f"[{exp_name}][VAL][PROTO] {results['prototype']['validation']}")

    for sp, loader in eval_loaders.items():
        scores, y = collect_prototype_scores(model, loader)
        results["prototype"][sp] = eval_metrics(scores, y, delta_proto)
        print(f"[{exp_name}][{sp}][PROTO] {results['prototype'][sp]}")

    # ---------- classifier branch ----------
    val_scores_cls, val_y_cls = collect_cls_scores(model, val_loader)
    delta_cls, J_cls = youden_best_threshold(val_scores_cls, val_y_cls)

    results["classifier"]["validation"] = {
        "threshold": delta_cls,
        "J": J_cls,
        **eval_metrics(val_scores_cls, val_y_cls, delta_cls),
    }

    print(f"[{exp_name}][VAL][CLS] best delta = {delta_cls:.6f}, J = {J_cls:.4f}")
    print(f"[{exp_name}][VAL][CLS] {results['classifier']['validation']}")

    for sp, loader in eval_loaders.items():
        scores, y = collect_cls_scores(model, loader)
        results["classifier"][sp] = eval_metrics(scores, y, delta_cls)
        print(f"[{exp_name}][{sp}][CLS] {results['classifier'][sp]}")

    return results


# =========================================================
# 10) Run an entire testbed
# =========================================================
def run_testbed(testbed, testbed_name="testbed", model_name=MODEL_NAME, epochs=EPOCHS):
    """
    Supports:
      A) testbed is a single DatasetDict
      B) testbed is a dict[str, DatasetDict]
    """
    all_results = {}

    if is_datasetdict_like(testbed):
        all_results["__single__"] = run_single_datasetdict_experiment(
            testbed,
            exp_name=testbed_name,
            model_name=model_name,
            epochs=epochs,
        )
        return all_results

    if is_mapping_of_datasetdict(testbed):
        for subset_name, ds_dict in testbed.items():
            exp_name = f"{testbed_name}/{subset_name}"
            all_results[subset_name] = run_single_datasetdict_experiment(
                ds_dict,
                exp_name=exp_name,
                model_name=model_name,
                epochs=epochs,
            )
        return all_results

    raise TypeError(
        "Unsupported testbed structure. "
        "Expected a DatasetDict-like object or a dict[str, DatasetDict]."
    )


# =========================================================
# 11) Optional summary printer
# =========================================================
def summarize_results(all_results):
    """
    Compact summary for quick comparison.
    """
    print("\n================ SUMMARY ================")
    for subset_name, res in all_results.items():
        meta = res["meta"]
        print(f"\n--- {meta['exp_name']} ---")
        print("machine_model_labels:", meta["machine_model_labels"])

        for branch_name in ["prototype", "classifier"]:
            print(f"  [{branch_name}]")
            for sp, metrics in res[branch_name].items():
                if sp == "validation":
                    print(
                        f"    {sp}: "
                        f"acc={metrics['acc']:.4f}, "
                        f"f1={metrics['f1']:.4f}, "
                        f"auroc={metrics['auroc']:.4f}, "
                        f"thr={metrics['threshold']:.6f}"
                    )
                else:
                    print(
                        f"    {sp}: "
                        f"acc={metrics['acc']:.4f}, "
                        f"f1={metrics['f1']:.4f}, "
                        f"auroc={metrics['auroc']:.4f}"
                    )

device: cuda


In [44]:
results_tb6 = run_testbed(testbed6, testbed_name="testbed6", epochs=EPOCHS)
summarize_results(results_tb6)


========== testbed6/excluded_cmv ==========
train label counter: Counter({0: 205365, 1: 89095})
train model_label counter: Counter({0: 89095, 5: 58884, 1: 47586, 4: 34317, 2: 27337, 6: 19950, 7: 10579, 3: 6712})
validation label counter: Counter({1: 26363, 0: 25496})
validation model_label counter: Counter({0: 26363, 5: 7322, 1: 5923, 4: 4258, 2: 3389, 6: 2483, 7: 1280, 3: 841})
test_id label counter: Counter({1: 26338, 0: 25564})
test_id model_label counter: Counter({0: 26338, 5: 7329, 1: 5929, 4: 4280, 2: 3398, 6: 2469, 7: 1320, 3: 839})
test_ood label counter: Counter({0: 2514, 1: 2403})
test_ood model_label counter: Counter({0: 2403, 1: 713, 5: 683, 4: 380, 2: 312, 6: 224, 7: 122, 3: 80})
Inferred machine model labels from TRAIN: [1, 2, 3, 4, 5, 6, 7]
Number of machine families: 7


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Initializing prototypes:   0%|          | 0/9202 [00:00<?, ?it/s]

Initialized human prototype from 89095 human samples
Machine model labels: [1, 2, 3, 4, 5, 6, 7]
Counts per machine prototype: [47586, 27337, 6712, 34317, 58884, 19950, 10579]


Train epoch 0:   0%|          | 0/9202 [00:00<?, ?it/s]

...

[testbed6/excluded_yelp][VAL][CLS] best delta = 0.034725, J = 0.8625
[testbed6/excluded_yelp][VAL][CLS] {'threshold': 0.03472512437525098, 'J': 0.8624874801230109, 'acc': 0.9309689554206821, 'human_recall': 0.9113686787544947, 'ai_recall': 0.9511188013685162, 'f1': 0.9304823276703769, 'auroc': 0.974420216772029}


Collect classifier-head scores:   0%|          | 0/1614 [00:00<?, ?it/s]

[testbed6/excluded_yelp][test_id][CLS] {'acc': 0.9321774506005425, 'human_recall': 0.9145616926674077, 'ai_recall': 0.9501782147193608, 'f1': 0.9316491282872259, 'auroc': 0.9750924470573359}


Collect classifier-head scores:   0%|          | 0/163 [00:00<?, ?it/s]

[testbed6/excluded_yelp][test_ood][CLS] {'acc': 0.8436237738026544, 'human_recall': 0.7349170437405732, 'ai_recall': 0.9568119356105221, 'f1': 0.8274251751220547, 'auroc': 0.9134983279651748}

================ SUMMARY ================

--- testbed6/excluded_cmv ---
machine_model_labels: [1, 2, 3, 4, 5, 6, 7]
  [prototype]
    validation: acc=0.9214, f1=0.9233, auroc=0.9745, thr=-0.260729
    test_id: acc=0.9206, f1=0.9225, auroc=0.9748
    test_ood: acc=0.9048, f1=0.9007, auroc=0.9670
  [classifier]
    validation: acc=0.9203, f1=0.9208, auroc=0.9704, thr=0.039812
    test_id: acc=0.9203, f1=0.9208, auroc=0.9707
    test_ood: acc=0.8977, f1=0.8910, auroc=0.9597

--- testbed6/excluded_eli5 ---
machine_model_labels: [1, 2, 3, 4, 5, 6, 7]
  [prototype]
    validation: acc=0.8999, f1=0.8985, auroc=0.9534, thr=-0.293857
    test_id: acc=0.8992, f1=0.8976, auroc=0.9542
    test_ood: acc=0.8481, f1=0.8330, auroc=0.9214
  [classifier]
    validation: acc=0.8826, f1=0.8767, auroc=0.9437, thr=0.

In [45]:
results_tb5 = run_testbed(testbed5, testbed_name="testbed5", epochs=EPOCHS)
summarize_results(results_tb5)


========== testbed5/excluded_LLaMA_set ==========
train label counter: Counter({0: 195908, 1: 93318})
train model_label counter: Counter({0: 93318, 5: 64415, 1: 53339, 4: 37425, 6: 21800, 7: 11585, 3: 7344})
validation label counter: Counter({1: 28799, 0: 24294})
validation model_label counter: Counter({0: 28799, 5: 8002, 1: 6632, 4: 4634, 6: 2707, 7: 1400, 3: 919})
test_id label counter: Counter({1: 28741, 0: 24368})
test_id model_label counter: Counter({0: 28741, 5: 8012, 1: 6642, 4: 4660, 6: 2693, 7: 1442, 3: 919})
test_ood label counter: Counter({1: 3710, 0: 3710})
test_ood model_label counter: Counter({0: 3710, 2: 3710})
Inferred machine model labels from TRAIN: [1, 3, 4, 5, 6, 7]
Number of machine families: 6


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Initializing prototypes:   0%|          | 0/9039 [00:00<?, ?it/s]

Initialized human prototype from 93318 human samples
Machine model labels: [1, 3, 4, 5, 6, 7]
Counts per machine prototype: [53339, 7344, 37425, 64415, 21800, 11585]


Train epoch 0:   0%|          | 0/9039 [00:00<?, ?it/s]

...

[testbed5/excluded_EleutherAI_set][test_ood][PROTO] {'acc': 0.9608183079056866, 'human_recall': 0.9320388349514563, 'ai_recall': 0.9895977808599168, 'f1': 0.9596572652624062, 'auroc': 0.9971544183702324}


Collect classifier-head scores:   0%|          | 0/1731 [00:00<?, ?it/s]

[testbed5/excluded_EleutherAI_set][VAL][CLS] best delta = 0.019099, J = 0.8816
[testbed5/excluded_EleutherAI_set][VAL][CLS] {'threshold': 0.019099146070950753, 'J': 0.8816293163097545, 'acc': 0.9409300982091277, 'human_recall': 0.943713323379284, 'ai_recall': 0.9379159929304705, 'f1': 0.9432220448393143, 'auroc': 0.9813664864800664}


Collect classifier-head scores:   0%|          | 0/1731 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
results_tb4 = run_testbed(testbed4, testbed_name="testbed4", epochs=EPOCHS)
summarize_results(results_tb4)

In [ ]:
results_tb3 = run_testbed(testbed3, testbed_name="testbed3", epochs=EPOCHS)
summarize_results(results_tb3)

In [ ]:
results_tb2 = run_testbed(testbed2, testbed_name="testbed2", epochs= EPOCHS)
summarize_results(results_tb2)

In [ ]:
results_tb1 = run_testbed(testbed1, testbed_name="testbed1", epochs=EPOCHS)
summarize_results(results_tb1)

In [39]:
results_tb1 = run_testbed(testbed1, testbed_name="testbed1", epochs=EPOCHS)
summarize_results(results_tb1)


========== testbed1/cmv ==========
train label counter: Counter({0: 512, 1: 438})
train model_label counter: Counter({7: 512, 0: 438})
validation label counter: Counter({0: 64, 1: 39})
validation model_label counter: Counter({7: 64, 0: 39})
test label counter: Counter({0: 60, 1: 32})
test model_label counter: Counter({7: 60, 0: 32})
Inferred machine model labels from TRAIN: [7]
Number of machine families: 1


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Initializing prototypes:   0%|          | 0/30 [00:00<?, ?it/s]

Initialized human prototype from 438 human samples
Machine model labels: [7]
Counts per machine prototype: [512]


Train epoch 0:   0%|          | 0/30 [00:00<?, ?it/s]

...

[testbed1/yelp][test][PROTO] {'acc': 0.9777777777777777, 'human_recall': 0.9897959183673469, 'ai_recall': 0.9634146341463414, 'f1': 0.9797979797979798, 'auroc': 0.9859382777501244}


Collect classifier-head scores:   0%|          | 0/6 [00:00<?, ?it/s]

[testbed1/yelp][VAL][CLS] best delta = 0.404564, J = 0.9524
[testbed1/yelp][VAL][CLS] {'threshold': 0.4045642922821148, 'J': 0.9523809523809523, 'acc': 0.9782608695652174, 'human_recall': 1.0, 'ai_recall': 0.9523809523809523, 'f1': 0.9803921568627451, 'auroc': 0.9951190476190476}


Collect classifier-head scores:   0%|          | 0/6 [00:00<?, ?it/s]

[testbed1/yelp][test][CLS] {'acc': 0.9833333333333333, 'human_recall': 1.0, 'ai_recall': 0.9634146341463414, 'f1': 0.9849246231155779, 'auroc': 0.9897959183673469}

================ SUMMARY ================

--- testbed1/cmv ---
machine_model_labels: [7]
  [prototype]
    validation: acc=0.9903, f1=0.9870, auroc=0.9952, thr=-0.003796
    test: acc=0.9783, f1=0.9677, auroc=0.9953
  [classifier]
    validation: acc=0.9903, f1=0.9873, auroc=0.9992, thr=0.417125
    test: acc=0.9891, f1=0.9841, auroc=1.0000

--- testbed1/eli5 ---
machine_model_labels: [7]
  [prototype]
    validation: acc=0.9890, f1=0.9895, auroc=0.9978, thr=-0.006857
    test: acc=0.9519, f1=0.9543, auroc=0.9946
  [classifier]
    validation: acc=0.9835, f1=0.9841, auroc=0.9921, thr=0.387424
    test: acc=0.9679, f1=0.9691, auroc=0.9955

--- testbed1/hswag ---
machine_model_labels: [7]
  [prototype]
    validation: acc=0.9891, f1=0.9901, auroc=0.9935, thr=-0.027849
    test: acc=1.0000, f1=1.0000, auroc=1.0000
  [classifi

In [40]:
results_tb2 = run_testbed(testbed2, testbed_name="testbed2", epochs= EPOCHS)
summarize_results(results_tb2)


========== testbed2/LLaMA_set ==========
train label counter: Counter({1: 29845, 0: 29845})
train model_label counter: Counter({0: 29845, 2: 29845})
validation label counter: Counter({1: 3699, 0: 3699})
validation model_label counter: Counter({0: 3699, 2: 3699})
test label counter: Counter({1: 3710, 0: 3710})
test model_label counter: Counter({0: 3710, 2: 3710})
Inferred machine model labels from TRAIN: [2]
Number of machine families: 1


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Initializing prototypes:   0%|          | 0/1866 [00:00<?, ?it/s]

Initialized human prototype from 29845 human samples
Machine model labels: [2]
Counts per machine prototype: [29845]


Train epoch 0:   0%|          | 0/1866 [00:00<?, ?it/s]

...

[testbed2/OPT_set][test][PROTO] {'acc': 0.9849600599101348, 'human_recall': 0.986145781328008, 'ai_recall': 0.9837743384922616, 'f1': 0.984977871969083, 'auroc': 0.9980657134617593}


Collect classifier-head scores:   0%|          | 0/501 [00:00<?, ?it/s]

[testbed2/OPT_set][VAL][CLS] best delta = 0.044754, J = 0.9699
[testbed2/OPT_set][VAL][CLS] {'threshold': 0.04475365777756512, 'J': 0.9698825293676581, 'acc': 0.984941264683829, 'human_recall': 0.9857535616095976, 'ai_recall': 0.9841289677580605, 'f1': 0.9849534869201474, 'auroc': 0.9983020834394002}


Collect classifier-head scores:   0%|          | 0/501 [00:00<?, ?it/s]

[testbed2/OPT_set][test][CLS] {'acc': 0.9849600599101348, 'human_recall': 0.986145781328008, 'ai_recall': 0.9837743384922616, 'f1': 0.984977871969083, 'auroc': 0.9980556187685521}

================ SUMMARY ================

--- testbed2/LLaMA_set ---
machine_model_labels: [2]
  [prototype]
    validation: acc=0.9374, f1=0.9380, auroc=0.9814, thr=-0.188324
    test: acc=0.9410, f1=0.9414, auroc=0.9850
  [classifier]
    validation: acc=0.9380, f1=0.9385, auroc=0.9814, thr=0.081883
    test: acc=0.9410, f1=0.9415, auroc=0.9852

--- testbed2/BigScience_set ---
machine_model_labels: [6]
  [prototype]
    validation: acc=0.9704, f1=0.9708, auroc=0.9958, thr=-0.155768
    test: acc=0.9699, f1=0.9702, auroc=0.9949
  [classifier]
    validation: acc=0.9701, f1=0.9704, auroc=0.9956, thr=0.123950
    test: acc=0.9697, f1=0.9699, auroc=0.9949

--- testbed2/FLAN_T5_set ---
machine_model_labels: [4]
  [prototype]
    validation: acc=0.9553, f1=0.9551, auroc=0.9917, thr=-0.053002
    test: acc=0.954

In [41]:
results_tb3 = run_testbed(testbed3, testbed_name="testbed3", epochs=EPOCHS)
summarize_results(results_tb3)


========== testbed3/cmv ==========
train label counter: Counter({0: 20388, 1: 4223})
train model_label counter: Counter({1: 5753, 5: 5531, 0: 4223, 4: 3108, 2: 2508, 6: 1850, 7: 1006, 3: 632})
validation label counter: Counter({0: 2497, 1: 2436})
validation model_label counter: Counter({0: 2436, 1: 709, 5: 680, 4: 376, 2: 310, 6: 224, 7: 120, 3: 78})
test label counter: Counter({0: 2514, 1: 2403})
test model_label counter: Counter({0: 2403, 1: 713, 5: 683, 4: 380, 2: 312, 6: 224, 7: 122, 3: 80})
Inferred machine model labels from TRAIN: [1, 2, 3, 4, 5, 6, 7]
Number of machine families: 7


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Initializing prototypes:   0%|          | 0/770 [00:00<?, ?it/s]

Initialized human prototype from 4223 human samples
Machine model labels: [1, 2, 3, 4, 5, 6, 7]
Counts per machine prototype: [5753, 2508, 632, 3108, 5531, 1850, 1006]


Train epoch 0:   0%|          | 0/770 [00:00<?, ?it/s]

...

[testbed3/sci_gen] epoch 2: {'loss': 6.260587762830004, 'contrast': 5.9092382619654655, 'ce': 0.35134949088921025}


Collect prototype scores:   0%|          | 0/152 [00:00<?, ?it/s]

[testbed3/sci_gen][VAL][PROTO] best delta = -0.045015, J = 0.8739
[testbed3/sci_gen][VAL][PROTO] {'threshold': -0.04501492274822496, 'J': 0.8738596421672301, 'acc': 0.9368160231261615, 'human_recall': 0.9344132753852232, 'ai_recall': 0.9394463667820069, 'f1': 0.9392374900714853, 'auroc': 0.9775761525936519}


Collect prototype scores:   0%|          | 0/150 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
results_tb4 = run_testbed(testbed4, testbed_name="testbed4", epochs=EPOCHS)
summarize_results(results_tb4)

In [ ]:
results_tb5 = run_testbed(testbed5, testbed_name="testbed5", epochs=EPOCHS)
summarize_results(results_tb5)

In [ ]:
results_tb6 = run_testbed(testbed6, testbed_name="testbed6", epochs=EPOCHS)
summarize_results(results_tb6)